In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 21 — VALIDAÇÃO CRUZADA ANINHADA: O TESTE DE ESTRESSE FINAL

> "Confiar na pontuação de um GridSearchCV é como deixar o aluno que fez a prova corrigi-la. Ele pode ter as respostas certas, mas sua nota será sempre otimista. Para uma avaliação imparcial, você precisa de um examinador externo."
> — Um Estatístico Cético

Eu tinha um modelo afinado, calibrado e explicável. O GridSearchCV me dizia esperar ~97,7% de acurácia. Eu me sentia confiante. Mas, no fundo, uma dúvida começou a se formar — não sobre o modelo, mas sobre o *processo* que me deu aquele número.

O GridSearchCV testou 64 combinações e reportou a pontuação da *melhor*. O problema estava nessa palavra: "melhor". Ele usou os mesmos dados para **escolher** os hiperparâmetros e para **avaliar** essa escolha. Isso introduz um viés de otimismo: a pontuação é a do campeão de um torneio interno, não a de como ele se sairia contra oponentes nunca vistos. Para uma estimativa verdadeiramente imparcial, eu precisava de um teste de estresse.

## O Viés Otimista da Otimização

Quando usamos `GridSearchCV`, os dados servem a dois propósitos: **selecionar** o modelo (achar os melhores hiperparâmetros) e **estimar** seu desempenho. Fazer as duas coisas com o mesmo procedimento de validação cruzada é uma falha metodológica: o desempenho reportado será otimista, porque a seleção já "viu" todos os dados e escolheu o que se ajusta melhor a eles.

A solução é a **Validação Cruzada Aninhada**, com dois laços:

**Laço externo** (avaliação) — divide os dados em *K* partes; cada uma serve, na sua vez, de teste.

**Laço interno** (seleção) — para cada divisão externa, um `GridSearchCV` completo roda **apenas** nos dados de treino daquela divisão, escolhendo hiperparâmetros que podem até variar entre as divisões.

A pontuação final é a média dos testes externos — uma estimativa do desempenho do **processo de modelagem**, não de um conjunto fixo de hiperparâmetros.

#### Código 21.1: Executando a Validação Cruzada Aninhada


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from oncoclassify_utils import X_train, y_train

def info_mutua(X, y): return mutual_info_classif(X, y, random_state=42)

pipeline = Pipeline([("selecao", SelectKBest(info_mutua)),
                     ("escala", StandardScaler()),
                     ("svm", SVC(kernel="rbf", random_state=42))])
grade = {"selecao__k": [15, 20, 25], "svm__C": [1, 10, 100],
         "svm__gamma": ["scale", 0.01, 0.001]}

laco_interno = StratifiedKFold(5, shuffle=True, random_state=42)  # SELECIONA
laco_externo = StratifiedKFold(5, shuffle=True, random_state=42)  # AVALIA

busca = GridSearchCV(pipeline, grade, cv=laco_interno, scoring="accuracy", n_jobs=-1)
scores = cross_val_score(busca, X_train, y_train, cv=laco_externo, n_jobs=-1)

print("Scores do laco externo:", np.round(scores, 4))
print(f"Acuracia ANINHADA (imparcial): {scores.mean():.4f} +/- {scores.std():.4f}")

Scores do laco externo: [0.9688 0.9896 0.9896 0.9583 0.9167]
Acuracia ANINHADA (imparcial): 0.9646 +/- 0.0268


O resultado é sóbrio e honesto. A estimativa **aninhada é 96,46%** (± 2,68%), **abaixo** dos 97,71% que o GridSearchCV simples reportou. A diferença não é enorme — nosso dataset é pequeno e bem-comportado —, mas é real: **o viés de otimismo existia**, e a validação aninhada o removeu. É esse 96,46% que eu reportaria ao chefe da oncologia como a expectativa honesta de desempenho do processo em dados futuros. Note também o desvio de ±2,7%: um lembrete de que, com 480 amostras, há variabilidade genuína entre as divisões.

> **📌 CHECKLIST DO CAPÍTULO 21**
>
> ✅ Entende por que a pontuação de um `GridSearchCV` é otimista.
>
> ✅ Sabe descrever a estrutura da validação aninhada (laço externo avalia, interno seleciona).
>
> ✅ Sabe que o resultado estima o desempenho do **processo**, não de um conjunto fixo de hiperparâmetros.
>
> ✅ Executou o Código 21.1 e comparou a estimativa imparcial (**96,46%**) com a otimista (97,71%).
>
> **⚠️ CRÍTICO:** A validação aninhada é o padrão-ouro para avaliação quando há otimização de hiperparâmetros. É cara, mas o custo se paga em confiança.

O resultado foi sóbrio, mas libertador. O número era um pouco menor, e eu confiava nele infinitamente mais. Eu havia separado a seleção da avaliação, o treinamento do exame final.

Com a metodologia validada, eu quis explorar uma última fronteira na busca por desempenho. Eu usara ensembles de árvores com sucesso. E se aplicasse a mesma filosofia aos meus melhores modelos de famílias diferentes — SVM, Regressão Logística, Gradient Boosting — e os fizesse votar? Era hora de reunir o comitê de especialistas.
